In [8]:
import os
import sys
import pandas as pd
import re
from collections import Counter

# 환경 설정
project_dir = "/data/ephemeral/home/nlp-5/eunbyul/joe"
sys.path.append(project_dir)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### 특수 토큰 등장 현황/문맥/치환 여부 확인 코드

In [ ]:
# (1) 특수 토큰 리스트
SPECIAL_TOKENS = [
    '#Person1#', '#Person2#', '#Person3#', '#Person4#', '#Person5#', '#Person6#', '#Person7#',
    '#PhoneNumber#', '#Address#', '#DateOfBirth#', '#PassportNumber#', '#SSN#', '#CardNumber#',
    '#CarNumber#', '#Email#', '#Name#', '#PersonName#', '#Price#', '#FlightNumber#',
    '#Alex#', '#Bob#', '#Kristin#', '#Liliana#', '#Mike#'
    # 필요하면 더 추가
]

# (2) 데이터 경로 지정 (알맞게 수정)
project_dir = "/data/ephemeral/home/nlp-5/eunbyul/joe"
data_dir = os.path.join(project_dir, 'data')

# (3) 데이터 로딩
datasets = {
    'train': pd.read_csv(os.path.join(data_dir, 'train.csv')),
    'dev': pd.read_csv(os.path.join(data_dir, 'dev.csv')),
    'test': pd.read_csv(os.path.join(data_dir, 'test.csv')),
}

SAMPLE_N = 3  # 샘플 몇 개 뽑을지

results = []
for dname, df in datasets.items():
    for token in SPECIAL_TOKENS:
        dialog_idx = df['dialogue'].str.contains(re.escape(token), na=False)

        # summary 컬럼이 있을 때만 summary_idx 계산
        if 'summary' in df.columns:
            summary_idx = df['summary'].str.contains(re.escape(token), na=False)
        else:
            summary_idx = None

        # 샘플 추출 (컬럼이 없으면 빈 문자열로)
        if dialog_idx.any():
            if 'summary' in df.columns:
                dialog_samples = df[dialog_idx][['dialogue', 'summary']].head(SAMPLE_N)
            else:
                dialog_samples = df[dialog_idx][['dialogue']].head(SAMPLE_N)
                dialog_samples['summary'] = ""
        else:
            dialog_samples = pd.DataFrame(columns=['dialogue', 'summary'])

        if summary_idx is not None and summary_idx.any():
            summary_samples = df[summary_idx][['dialogue', 'summary']].head(SAMPLE_N)
        else:
            summary_samples = pd.DataFrame(columns=['dialogue', 'summary'])

        results.append({
            "dataset": dname,
            "token": token,
            "dialogue_count": int(dialog_idx.sum()),
            "summary_count": int(summary_idx.sum() if summary_idx is not None else 0),
            "dialogue_sample_1": dialog_samples['dialogue'].iloc[0] if len(dialog_samples) > 0 else "",
            "dialogue_sample_1_summary": dialog_samples['summary'].iloc[0] if len(dialog_samples) > 0 else "",
            "dialogue_sample_2": dialog_samples['dialogue'].iloc[1] if len(dialog_samples) > 1 else "",
            "dialogue_sample_2_summary": dialog_samples['summary'].iloc[1] if len(dialog_samples) > 1 else "",
            "dialogue_sample_3": dialog_samples['dialogue'].iloc[2] if len(dialog_samples) > 2 else "",
            "dialogue_sample_3_summary": dialog_samples['summary'].iloc[2] if len(dialog_samples) > 2 else "",
            "summary_sample_1": summary_samples['summary'].iloc[0] if len(summary_samples) > 0 else "",
            "summary_sample_1_dialogue": summary_samples['dialogue'].iloc[0] if len(summary_samples) > 0 else "",
            "summary_sample_2": summary_samples['summary'].iloc[1] if len(summary_samples) > 1 else "",
            "summary_sample_2_dialogue": summary_samples['dialogue'].iloc[1] if len(summary_samples) > 1 else "",
            "summary_sample_3": summary_samples['summary'].iloc[2] if len(summary_samples) > 2 else "",
            "summary_sample_3_dialogue": summary_samples['dialogue'].iloc[2] if len(summary_samples) > 2 else "",
        })

# (4) DataFrame으로 변환 및 csv 저장
out_df = pd.DataFrame(results)
out_csv_path = os.path.join(project_dir, "src/notebooks/special_token_analysis.csv")
out_df.to_csv(out_csv_path, index=False)
print(f"\n분석 결과를 저장했습니다: {out_csv_path}")


분석 결과를 저장했습니다: /data/ephemeral/home/nlp-5/eunbyul/joe/src/notebooks/special_token_analysis.csv
   dataset             token  dialogue_count  summary_count  \
0    train         #Person1#           12457          10095   
1    train         #Person2#           12457           8768   
2    train         #Person3#             122             57   
3    train         #Person4#              15              4   
4    train         #Person5#               5              0   
5    train         #Person6#               2              1   
6    train         #Person7#               1              1   
7    train     #PhoneNumber#             151              2   
8    train         #Address#              81              4   
9    train     #DateOfBirth#              20              0   
10   train  #PassportNumber#               5              0   
11   train             #SSN#               3              0   
12   train      #CardNumber#              12              0   
13   train       #CarN

In [22]:
import pandas as pd
import os

data_dir = '/data/ephemeral/home/nlp-5/eunbyul/joe/data'  # 본인 경로로 수정
df = pd.read_csv(os.path.join(data_dir, 'train.csv'))  # 쉼표 삭제!

# 예시: “그 사람”, “이것”, “저기” 등 자연어 지시어의 빈도
for word in ['그 사람', '이것', '저기', '이 사람']:
    print(f'"{word}" in summary:', df['summary'].str.contains(word).sum())

# 예시: “#지시_사람#”, “#지시_사물#”, “#Person1#” 같은 의미 토큰의 빈도
for token in ['#지시_사람#', '#지시_사물#', '#Person1#']:
    print(f'"{token}" in summary:', df['summary'].str.contains(token).sum())

# summary 샘플 30개 직접 눈으로 확인
print(df['summary'].sample(30).tolist())


"그 사람" in summary: 14
"이것" in summary: 12
"저기" in summary: 2
"이 사람" in summary: 9
"#지시_사람#" in summary: 0
"#지시_사물#" in summary: 0
"#Person1#" in summary: 10095
['Mrs. Bird는 남편이 스테이크를 좋아해서 #Person1#에게 소고기와 스테이크를 구매합니다.', '#Person1#은 Dr. Smith와의 예약을 다음 주 화요일 오후 3시로 변경했습니다.', 'Chris는 임신한 아내 때문에 차를 사려고 고민 중인데, 큰 재정적 부담이라고 생각하고 있다. #Person1#도 자신이 차를 사기 위해 돈을 모았던 경험 때문에 같은 생각이다. Chris는 지프를 선호하지만 아내는 세단을 선호하고, #Person1#은 아내의 말을 따르라고 조언한다.', '안나는 조던과 헤어지기로 하고, 조던은 이를 받아들인다. 둘은 친구 관계를 유지하기로 한다.', '#Person1#는 Markheed Inc.의 일자리를 고민 중이다. #Person2#는 #Person1#에게 훌륭한 복지 혜택, 적정한 급여 수준, 좋은 근무 환경 등의 장점을 설명하며, 그 일을 받아들여야 한다고 조언한다.', '#Person2#는 #Person1#에게 컴퓨터 관련 경험과 NCRE 자격증이 있지만, 관리 정보 처리에는 익숙하지 않다고 설명합니다.', '인터뷰 중에 #Person2#는 자신의 약점, 협동 능력, 영어 실력, 그리고 출장 가능성에 대해 설명합니다.', '#Person1#은 며칠 동안 인터넷 연결이 안 되어 수리를 요청했으며, #Person2#는 전문가를 보내 해결할 예정입니다.', '#Person1#이 이 과장님과 그의 부인을 정식 앉아서 하는 연말 저녁식사에 초대했으며, 참석자들은 모두 회사 동료들입니다. 이 과장님은 기대감을 보였습니다.', '#Person1#은 거래 후 가격이 떨어졌을 때 어떻게 해야 할지 문의합니다. #Person2#는 거래하는 물품에 변

In [ ]:
import pandas as pd
import re
from collections import Counter

data_dir = '/data/ephemeral/home/nlp-5/eunbyul/joe/data'
files = ['train.csv', 'dev.csv', 'test.csv']

# 네가 만든 DEICTIC_NOUNS
DEICTIC_NOUNS = [
    # 아래 리스트는 네가 이미지로 준 "5회 이상" 등장 빈도 기준으로 추출함
    # 실제 프로젝트에선 통계 기반으로 더 확장/조정 가능
    "그게","그런","그거","그건","그녀","저기","그 사람","이거","그때","이건","그걸","그분","이게","이 집","그곳",
    "그것","그 친구","그쪽","이걸","그의","그 얘기","그 일","이 다음","저","이곳","저 여자","이 회사","이 차",
    "이것","그녀의","저쪽","그","그날","그 말","그 사람들","이 도시","그 돈","이 책","그 애","이 시간","그 문제",
    "이 문제","그다음에","이 지역","그 사람의","그 남자","저번","이 일","이 버스","그 수업","이 약","그 책",
    "그 자리","저 사람","이 셔츠","이쪽","이 선물","이 자리","그 후","그 정보","그 영화","그분들","이 날","저거",
    "이 새","이 요즘","그 회사","저건","이 신발","이 책들","이 사람들","저의","그 차","이 드레스","이 길","저요",
    "이 수업","그 정도면","그 전","그 이후에","이 보고서","그 시간","이 선생님","그 부분","그 여자","그 다음엔",
    "이 방","이 가게","이 초콜릿","이 근처","이 도움","그 가격","이 소포","그 기간","이 서류","이 영화","그것들",
    "이 색","이 계약","이 주소","저게","그 길","그 이후","그 순간","그 표지판","그 근처","이 지역의","이 음악",
    "이 가방","이 동네","이 모델","이 스타일","이 돈","그 생각","이곳의","이 가격","저 건물","이디어","이 바지",
    "그분의","이대","그림","그 집","이 서비스","그 점","그다음","그 외에","이 코트","그 마음","이후에","이 정도면",
    "그러시군요","이 노래","이 처방전","저 길","이 옷","이 문","이 주말","이 반지","이 사진","이 여기","그 후에",
    "그 다음에","저 ","그 외","그 다음","이 편지","그 정","이 선반","이 공","그 중","이 지금","이 제품","이 사람",
    "이 직업","이 시간대에","그 둘","이 그","그 분야","이 꼭","이블","이웃","그땐","이 학교","이 양식","이 종이",
    "이 분야","이 줄","이 생각해","그곳의","이 사무실","그중","이 회사의","이 기계","그 잡지","그 나라의","그 전에",
    "그 버스","이 신문","이 모두","이 스웨터","이 운동","이 우리","이 나라","이 제안","그 색상","이 기간","이 아파트",
    "이 도시의","그 드레스","이 신청서","이 음식","이 건물","이 때문","그 가격대",
    "이 직책","이 프로젝트","이 도시에서","그 학생","이 그림","그 나라","이룰","그 식당","이웃들","이 하루","그 이후로",
    "이 여기서","이 얘기","그 선생님","이 인기","그 소식","이 실제","이 직무","이 첫","그 노래","이 정","그 비행기",
    "이 비용","이 완벽","그 안","이 말","이 분야의","그 목표","이 거리","그 분","이 세계","이 광고","그 뒤","그 정도",
    "이 너희","이 컴퓨터","이 기회","이 환전","이 대부분","이 치마","그 얘긴","이 잡지","그 학교",
    "그 선물","이 경기","이 프로그램","이 서류들","그 이야기","이 나무"
]

SAMPLE_N = 3
results = []
for fname in files:
    df = pd.read_csv(os.path.join(data_dir, fname))
    has_summary = 'summary' in df.columns
    for noun in sorted(DEICTIC_NOUNS, key=len, reverse=True):
        if 'dialogue' in df.columns:
            idx = df['dialogue'].str.contains(re.escape(noun), na=False)
            cnt = idx.sum()
            if cnt > 0:
                # summary 없으면 빈 칸으로 대체
                if has_summary:
                    samples = df[idx][['dialogue', 'summary']].head(SAMPLE_N)
                else:
                    samples = df[idx][['dialogue']].head(SAMPLE_N)
                    samples['summary'] = ""  # summary 컬럼 추가

                for i, row in samples.iterrows():
                    results.append({
                        "file": fname,
                        "noun": noun,
                        "dialogue": row['dialogue'],
                        "summary": row['summary']
                    })

out_df = pd.DataFrame(results)
out_csv_path = os.path.join(project_dir, "src/notebooks/deictic_noun_samples_dialogue_summary.csv")
out_df.to_csv(out_csv_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {out_csv_path}")
print(out_df.head(10))

저장 완료: /data/ephemeral/home/nlp-5/eunbyul/joe/src/notebooks/deictic_noun_samples_dialogue_summary.csv
        file    noun                                           dialogue  \
0  train.csv  이 시간대에  #Person1#: 안녕하세요, 택시 기사님. 금융가로 가주세요.\n#Person2...   
1  train.csv  이 시간대에  #Person1#: 저기요, 여기서 박물관 가는 버스를 탈 수 있나요?\n#Pers...   
2  train.csv  이 시간대에  #Person1#: 버스를 타고 왕푸징에 가자.\n#Person2#: 지하철 타는 ...   
3  train.csv  이 프로젝트  #Person1#: 좋아요. 이 프로젝트에 모두 함께 하고 싶어요. 언제 시작할 수...   
4  train.csv  이 프로젝트  #Person1#: 도와드릴 일 있나요?\n#Person2#: 네, 감사합니다. 제...   
5  train.csv  이 프로젝트  #Person1#: 기분이 좋아 보이지 않는데, 무슨 일 있어?\n#Person2#...   
6  train.csv  이 도시에서  #Person1#: 택시! 아, 고마워요, 세워주셔서. \n#Person2#: 어디...   
7  train.csv  이 도시에서  #Person1#: 아파트 알아보려면 어디서 시작해야 할까? \n#Person2#:...   
8  train.csv  이 도시에서  #Person1#: 저기요, 버스 노선도 받을 수 있을까요?\n#Person2#: ...   
9  train.csv  이 프로그램  #Person1#: 저기요, 동물원 가는 길 좀 알려주시겠어요?\n#Person2#...   

                                             summary  
0  #Person1#은 금융가

In [ ]:
import pandas as pd
import re
from collections import Counter

# 파일 경로
test_path = "/data/ephemeral/home/nlp-5/eunbyul/joe/data/test.csv"  # 실제 경로에 맞게 변경
pred_path = "/data/ephemeral/home/nlp-5/eunbyul/joe/prediction/submission_digit82-kobart-summarization_beam4_len165_20250803_2326.csv"

# 데이터 로드
test_df = pd.read_csv(test_path)
pred_df = pd.read_csv(pred_path)

# 분석할 패턴
DEICTIC_PATTERNS = [
    r'\b[이그저]\s?[가-힣]{1,10}\b',  # 이/그/저 + 명사
    r'그 사람', r'이 사람', r'저 사람'
]

# 집계 함수
def count_patterns(texts, patterns):
    total = Counter()
    for txt in texts:
        for patt in patterns:
            found = re.findall(patt, str(txt))
            total.update(found)
    return total

# 1. test.csv의 dialogue 전체에서 패턴 집계
dialogue_counter = count_patterns(test_df['dialogue'], DEICTIC_PATTERNS)

# 2. 제출 summary 전체에서 패턴 집계
summary_counter = count_patterns(pred_df['summary'], DEICTIC_PATTERNS)

print("===== test.csv dialogue 내 패턴 빈도 =====")
for k, v in dialogue_counter.most_common(30):
    print(f"{k} : {v}")
print("\n===== 제출 summary 내 패턴 빈도 =====")
for k, v in summary_counter.most_common(30):
    print(f"{k} : {v}")

===== test.csv dialogue 내 패턴 빈도 =====
그럼 : 162
그리고 : 116
그래 : 99
이제 : 87
그냥 : 79
그렇게 : 73
그게 : 72
그래서 : 69
그런데 : 65
저는 : 64
그건 : 59
그거 : 58
그런 : 57
이번 : 41
그러면 : 40
이렇게 : 30
저녁 : 29
이거 : 27
이런 : 27
이건 : 25
그럴 : 24
그래도 : 23
저희 : 23
저기 : 20
그녀가 : 20
저도 : 20
이게 : 19
이미 : 19
저희는 : 18
저기요 : 18

===== 제출 summary 내 패턴 빈도 =====
그들은 : 24
이야기합니다 : 20
이야기한다 : 11
이야기하고 : 10
저녁에 : 8
그리고 : 5
이번 : 5
저녁 : 5
그녀의 : 4
그는 : 4
그녀가 : 4
저녁을 : 4
그의 : 3
이를 : 3
이는 : 3
그것이 : 3
이유를 : 2
그녀는 : 2
이메일 : 2
이유와 : 2
이해하지 : 2
이름을 : 2
이용하면 : 1
이혼한 : 1
이혼이 : 1
이루어질 : 1
이 미디엄 : 1
이 펜고리를 : 1
이기는 : 1
이긴다고 : 1
